In [1]:
import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


MODEL_DIR_BASE = "saved_models"
RANDOM_STATE = 42

CICIDS_PATH = "/kaggle/input/datasets/steam2312/idsdatasets/cicids_supervised.csv"
CERT_PATH   = "/kaggle/input/datasets/steam2312/datasetss/cert_final.csv"
HDFS_PATH   = "/kaggle/input/datasets/steam2312/datasetss/hdfs_final.csv"

GLOBAL_WEIGHTS = {"CICIDS": 0.50, "CERT": 0.25, "HDFS": 0.25}
GLOBAL_THRESHOLD = 0.5


def preprocess_data(df, label_col, name):
    print(f"\n===== {name} =====")
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    df = df.loc[:, df.nunique() > 1]
    df[label_col] = df[label_col].astype(str).str.lower()
    df[label_col] = df[label_col].apply(lambda x: 0 if x in ['benign', 'normal', '0'] else 1)
    print("\nLabel Distribution:")
    print(df[label_col].value_counts())
    X = pd.get_dummies(df.drop(label_col, axis=1), drop_first=True)
    return X, df[label_col]


def split_data(X, y):
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.8, stratify=y_temp, random_state=RANDOM_STATE)
    return X_train, X_val, X_test, y_train, y_val, y_test

def train_models(X_train, y_train, name):
    print("\nTraining Random Forest...")
    rf = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                                random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)

    print("\nTraining Isolation Forest ONLY on NORMAL traffic")
    X_train_normal = X_train[y_train == 0]
    print("Normal Samples Used:", len(X_train_normal))

    scaler = StandardScaler()
    X_train_normal_scaled = scaler.fit_transform(X_train_normal)

    iso = IsolationForest(contamination=0.05, n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    iso.fit(X_train_normal_scaled)

    train_iso_scores = -iso.score_samples(X_train_normal_scaled)
    return rf, iso, scaler, train_iso_scores.min(), train_iso_scores.max()


def get_rf_scores(rf, X):
    return rf.predict_proba(X)[:, 1]

def get_iso_scores(iso, scaler, X, iso_min, iso_max):
    X_scaled = scaler.transform(X)
    iso_scores = -iso.score_samples(X_scaled)
    iso_scores = (iso_scores - iso_min) / (iso_max - iso_min + 1e-8)
    return np.clip(iso_scores, 0, 1)

def get_hybrid_scores(rf_scores, iso_scores, rf_weight, iso_weight):
    return rf_weight * rf_scores + iso_weight * iso_scores

def tune_threshold(scores, y):
    best_f1, best_th = 0, 0.5
    for t in np.linspace(0.1, 0.9, 100):
        preds = (scores > t).astype(int)
        f1 = f1_score(y, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = t
    return best_th

def find_best_weights(rf_scores, iso_scores, y_val):
    best_f1 = 0
    best_rf_w = best_iso_w = best_th = 0.5
    for rf_w in np.arange(0.0, 1.01, 0.01):
        iso_w = 1 - rf_w
        hybrid = get_hybrid_scores(rf_scores, iso_scores, rf_w, iso_w)
        th = tune_threshold(hybrid, y_val)
        f1 = f1_score(y_val, (hybrid > th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_rf_w, best_iso_w, best_th = rf_w, iso_w, th
    return best_rf_w, best_iso_w, best_th


def evaluate_model(y_true, y_pred, model_name):
    print("\n==============================")
    print(model_name)
    print("==============================")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall   :", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1 Score :", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


def run_pipeline(path, name):
    print(f"\n🔷 RUNNING {name} PIPELINE")
    df = pd.read_csv(path)
    if "label" not in df.columns:
        df = pd.read_csv(path, header=None)
        df.columns = [f"feature_{i}" for i in range(df.shape[1])]
        df.rename(columns={df.columns[-1]: "label"}, inplace=True)

    X, y = preprocess_data(df, "label", name)
    feature_columns = X.columns.tolist()
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    rf, iso, scaler, iso_min, iso_max = train_models(X_train, y_train, name)

    rf_val_scores = get_rf_scores(rf, X_val)
    iso_val_scores = get_iso_scores(iso, scaler, X_val, iso_min, iso_max)

    rf_weight, iso_weight, hybrid_threshold = find_best_weights(rf_val_scores, iso_val_scores, y_val)
    rf_threshold = tune_threshold(rf_val_scores, y_val)
    iso_threshold = tune_threshold(iso_val_scores, y_val)

    print("\nBest RF Threshold    :", round(rf_threshold, 4))
    print("Best ISO Threshold   :", round(iso_threshold, 4))
    print("Best Hybrid Threshold:", round(hybrid_threshold, 4))
    print("Best RF Weight       :", round(rf_weight, 4))
    print("Best ISO Weight      :", round(iso_weight, 4))

    
    rf_test_scores = get_rf_scores(rf, X_test)
    iso_test_scores = get_iso_scores(iso, scaler, X_test, iso_min, iso_max)
    hybrid_test_scores = get_hybrid_scores(rf_test_scores, iso_test_scores, rf_weight, iso_weight)

    rf_pred = (rf_test_scores > rf_threshold).astype(int)
    iso_pred = (iso_test_scores > iso_threshold).astype(int)
    hybrid_pred = (hybrid_test_scores > hybrid_threshold).astype(int)

    evaluate_model(y_test, rf_pred, f"{name} SUPERVISED MODEL (RF)")
    evaluate_model(y_test, iso_pred, f"{name} UNSUPERVISED MODEL (ISO)")
    evaluate_model(y_test, hybrid_pred, f"{name} OPTIMIZED HYBRID IDS")

    
    model_dir = f"{MODEL_DIR_BASE}/{name.lower()}"
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump({"rf_model": rf, "iso_model": iso, "scaler": scaler,
                 "rf_weight": rf_weight, "iso_weight": iso_weight,
                 "rf_threshold": rf_threshold, "iso_threshold": iso_threshold,
                 "hybrid_threshold": hybrid_threshold, "feature_columns": feature_columns,
                 "iso_min": iso_min, "iso_max": iso_max}, f"{model_dir}/results.pkl")

    return joblib.load(f"{model_dir}/results.pkl")


def global_weighted_scoring(cicids_h, cert_h, hdfs_h):
    final_score = (GLOBAL_WEIGHTS["CICIDS"] * cicids_h +
                   GLOBAL_WEIGHTS["CERT"] * cert_h +
                   GLOBAL_WEIGHTS["HDFS"] * hdfs_h)
    final_pred = "ATTACK" if final_score > GLOBAL_THRESHOLD else "NORMAL"

    print("\n" + "="*70)
    print(" GLOBAL ENSEMBLE SCORING (Weighted Sum of 3 Hybrids)")
    print("="*70)
    print(f"CICIDS Hybrid Score : {cicids_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['CICIDS']})")
    print(f"CERT   Hybrid Score : {cert_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['CERT']})")
    print(f"HDFS   Hybrid Score : {hdfs_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['HDFS']})")
    print("-" * 50)
    print(f"FINAL GLOBAL SCORE  : {final_score*100:.2f}%")
    print(f"Global Threshold    : {GLOBAL_THRESHOLD}")
    print(f"FINAL PREDICTION    : {final_pred}")
    print("="*70)
    return final_score, final_pred


cicids_results = run_pipeline(CICIDS_PATH, "CICIDS")
cert_results = run_pipeline(CERT_PATH, "CERT")
hdfs_results = run_pipeline(HDFS_PATH, "HDFS")

print("\nAll models trained successfully!")
print("Use predict_single_log() to test new logs.")


cicids_h = 0.2233
cert_h = 0.6267
hdfs_h = 0.5039

global_weighted_scoring(cicids_h, cert_h, hdfs_h)



def load_model(name):
    model_path = f"{MODEL_DIR_BASE}/{name.lower()}/results.pkl"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    return joblib.load(model_path)


def compute_global_metrics_test_split():
    """Compute metrics for each hybrid + overall summary"""
    print("\n" + "="*90)
    print("🌍 GLOBAL ENSEMBLE PERFORMANCE METRICS (TEST SPLIT ONLY)")
    print("="*90)
    
    # Load trained models
    cicids = load_model("CICIDS")
    cert   = load_model("CERT")
    hdfs   = load_model("HDFS")
    
    # Reload original datasets
    df_cicids = pd.read_csv(CICIDS_PATH)
    df_cert   = pd.read_csv(CERT_PATH)
    df_hdfs   = pd.read_csv(HDFS_PATH)
    
    # Preprocess
    X_cicids, y_cicids = preprocess_data(df_cicids.copy(), "label", "CICIDS")
    X_cert,   y_cert   = preprocess_data(df_cert.copy(),   "label", "CERT")
    X_hdfs,   y_hdfs   = preprocess_data(df_hdfs.copy(),   "label", "HDFS")
    
    # Get Test Split
    def get_test_split(X, y):
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.8, stratify=y_temp, random_state=RANDOM_STATE)
        return X_test, y_test
    
    X_test_c, y_test_c = get_test_split(X_cicids, y_cicids)
    X_test_ce, y_test_ce = get_test_split(X_cert, y_cert)
    X_test_h, y_test_h = get_test_split(X_hdfs, y_hdfs)
    
    # Compute Hybrid Scores for each dataset separately
    def get_hybrid_on_test(model, X_test):
        rf_s = get_rf_scores(model["rf_model"], X_test)
        iso_s = get_iso_scores(model["iso_model"], model["scaler"], X_test, 
                              model["iso_min"], model["iso_max"])
        return get_hybrid_scores(rf_s, iso_s, model["rf_weight"], model["iso_weight"])
    
    cicids_h = get_hybrid_on_test(cicids, X_test_c)
    cert_h   = get_hybrid_on_test(cert, X_test_ce)
    hdfs_h   = get_hybrid_on_test(hdfs, X_test_h)
    
    # Per-Dataset Predictions & Metrics
    def evaluate_hybrid(y_true, hybrid_scores, name):
        pred = (hybrid_scores > GLOBAL_THRESHOLD).astype(int)   # Using global threshold for consistency
        print(f"\n{name} Hybrid (on Test Set):")
        print("Accuracy  :", round(accuracy_score(y_true, pred), 4))
        print("Precision :", round(precision_score(y_true, pred, zero_division=0), 4))
        print("Recall    :", round(recall_score(y_true, pred, zero_division=0), 4))
        print("F1 Score  :", round(f1_score(y_true, pred, zero_division=0), 4))
        return accuracy_score(y_true, pred), f1_score(y_true, pred, zero_division=0)
    
    print("-"*80)
    acc_c, f1_c = evaluate_hybrid(y_test_c, cicids_h, "CICIDS")
    acc_ce, f1_ce = evaluate_hybrid(y_test_ce, cert_h, "CERT")
    acc_h, f1_h = evaluate_hybrid(y_test_h, hdfs_h, "HDFS")
    
    # Overall Average Metrics (Weighted by global weights)
    weighted_acc = (GLOBAL_WEIGHTS["CICIDS"]*acc_c + GLOBAL_WEIGHTS["CERT"]*acc_ce + GLOBAL_WEIGHTS["HDFS"]*acc_h)
    weighted_f1  = (GLOBAL_WEIGHTS["CICIDS"]*f1_c + GLOBAL_WEIGHTS["CERT"]*f1_ce + GLOBAL_WEIGHTS["HDFS"]*f1_h)
    
    print("\n" + "="*60)
    print("🌍 WEIGHTED GLOBAL ENSEMBLE SUMMARY")
    print("="*60)
    print("Weighted Average Accuracy :", round(weighted_acc, 4))
    print("Weighted Average F1 Score :", round(weighted_f1, 4))
    print(f"Global Threshold Used     : {GLOBAL_THRESHOLD}")
    print("="*90)
    
    return {"weighted_accuracy": round(weighted_acc, 4), "weighted_f1": round(weighted_f1, 4)}


# ====================== RUN IT ======================
if __name__ == "__main__":
    global_metrics = compute_global_metrics_test_split()
    print("\n✅ Global Ensemble Evaluation Completed!")


🔷 RUNNING CICIDS PIPELINE

===== CICIDS =====

Label Distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Training Random Forest...

Training Isolation Forest ONLY on NORMAL traffic
Normal Samples Used: 3750

Best RF Threshold    : 0.2455
Best ISO Threshold   : 0.3182
Best Hybrid Threshold: 0.2859
Best RF Weight       : 0.71
Best ISO Weight      : 0.29

CICIDS SUPERVISED MODEL (RF)
Accuracy : 0.9975
Precision: 0.997
Recall   : 0.998
F1 Score : 0.9975

Confusion Matrix:
[[997   3]
 [  2 998]]

CICIDS UNSUPERVISED MODEL (ISO)
Accuracy : 0.8725
Precision: 0.8892
Recall   : 0.851
F1 Score : 0.8697

Confusion Matrix:
[[894 106]
 [149 851]]

CICIDS OPTIMIZED HYBRID IDS
Accuracy : 0.997
Precision: 0.996
Recall   : 0.998
F1 Score : 0.997

Confusion Matrix:
[[996   4]
 [  2 998]]

🔷 RUNNING CERT PIPELINE

===== CERT =====

Label Distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Training Random Forest...

Training Isolation Forest ONLY on NORMAL traffic
Normal Sam

In [2]:
import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


MODEL_DIR_BASE = "saved_models"
RANDOM_STATE = 42

CICIDS_PATH = "/kaggle/input/datasets/steam2312/idsdatasets/cicids_supervised.csv"
CERT_PATH   = "/kaggle/input/datasets/steam2312/datasetss/cert_final.csv"
HDFS_PATH   = "/kaggle/input/datasets/steam2312/datasetss/hdfs_final.csv"

GLOBAL_WEIGHTS  = {"CICIDS": 0.50, "CERT": 0.25, "HDFS": 0.25}
GLOBAL_THRESHOLD = 0.5



def preprocess_data(df, label_col, name):
    print(f"\n===== {name} =====")
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    df = df.loc[:, df.nunique() > 1]
    df[label_col] = df[label_col].astype(str).str.lower()
    df[label_col] = df[label_col].apply(lambda x: 0 if x in ['benign', 'normal', '0'] else 1)
    print("\nLabel Distribution:")
    print(df[label_col].value_counts())
    X = pd.get_dummies(df.drop(label_col, axis=1), drop_first=True)
    return X, df[label_col]




def split_data(X, y):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.80, stratify=y_temp, random_state=RANDOM_STATE)
    return X_train, X_val, X_test, y_train, y_val, y_test



def train_models(X_train, y_train, name):
    print("\nTraining Random Forest...")
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)

    print("\nTraining Isolation Forest ONLY on NORMAL traffic")
    X_train_normal = X_train[y_train == 0]
    print("Normal Samples Used:", len(X_train_normal))

    scaler = StandardScaler()
    X_train_normal_scaled = scaler.fit_transform(X_train_normal)

    iso = IsolationForest(
        contamination=0.05, n_estimators=200,
        random_state=RANDOM_STATE, n_jobs=-1)
    iso.fit(X_train_normal_scaled)

    train_iso_scores = -iso.score_samples(X_train_normal_scaled)
    return rf, iso, scaler, train_iso_scores.min(), train_iso_scores.max()



def get_rf_scores(rf, X):
    return rf.predict_proba(X)[:, 1]


def get_iso_scores(iso, scaler, X, iso_min, iso_max):
    X_scaled    = scaler.transform(X)
    iso_scores  = -iso.score_samples(X_scaled)
    iso_scores  = (iso_scores - iso_min) / (iso_max - iso_min + 1e-8)
    return np.clip(iso_scores, 0, 1)


def get_hybrid_scores(rf_scores, iso_scores, rf_weight, iso_weight):
    return rf_weight * rf_scores + iso_weight * iso_scores



def tune_threshold(scores, y):
    best_f1, best_th = 0, 0.5
    for t in np.linspace(0.1, 0.9, 100):
        preds = (scores > t).astype(int)
        f1    = f1_score(y, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, t
    return best_th


def find_best_weights(rf_scores, iso_scores, y_val):
    best_f1 = 0
    best_rf_w = best_iso_w = best_th = 0.5
    for rf_w in np.arange(0.0, 1.01, 0.01):
        iso_w  = 1 - rf_w
        hybrid = get_hybrid_scores(rf_scores, iso_scores, rf_w, iso_w)
        th     = tune_threshold(hybrid, y_val)
        f1     = f1_score(y_val, (hybrid > th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_rf_w, best_iso_w, best_th = rf_w, iso_w, th
    return best_rf_w, best_iso_w, best_th



def evaluate_model(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print("\n" + "="*50)
    print(model_name)
    print("="*50)
    print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  Accuracy  : {round(accuracy_score(y_true, y_pred), 4)}")
    print(f"  Precision : {round(precision_score(y_true, y_pred, zero_division=0), 4)}")
    print(f"  Recall    : {round(recall_score(y_true, y_pred, zero_division=0), 4)}")
    print(f"  F1 Score  : {round(f1_score(y_true, y_pred, zero_division=0), 4)}")
    print("\n  Confusion Matrix:")
    print(f"  {cm}")



def run_pipeline(path, name):
    print(f"\n{'='*60}")
    print(f"  RUNNING {name} PIPELINE")
    print(f"{'='*60}")

    df = pd.read_csv(path)
    if "label" not in df.columns:
        df = pd.read_csv(path, header=None)
        df.columns = [f"feature_{i}" for i in range(df.shape[1])]
        df.rename(columns={df.columns[-1]: "label"}, inplace=True)

    X, y = preprocess_data(df, "label", name)
    feature_columns = X.columns.tolist()

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
    print(f"\n  Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

    rf, iso, scaler, iso_min, iso_max = train_models(X_train, y_train, name)

   
    rf_val_scores  = get_rf_scores(rf, X_val)
    iso_val_scores = get_iso_scores(iso, scaler, X_val, iso_min, iso_max)

    rf_weight, iso_weight, hybrid_threshold = find_best_weights(
        rf_val_scores, iso_val_scores, y_val)
    rf_threshold  = tune_threshold(rf_val_scores,  y_val)
    iso_threshold = tune_threshold(iso_val_scores, y_val)

    print(f"\n  Best RF Threshold     : {round(rf_threshold,      4)}")
    print(f"  Best ISO Threshold    : {round(iso_threshold,     4)}")
    print(f"  Best Hybrid Threshold : {round(hybrid_threshold,  4)}")
    print(f"  Best RF Weight        : {round(rf_weight,         4)}")
    print(f"  Best ISO Weight       : {round(iso_weight,        4)}")


    rf_test_scores     = get_rf_scores(rf, X_test)
    iso_test_scores    = get_iso_scores(iso, scaler, X_test, iso_min, iso_max)
    hybrid_test_scores = get_hybrid_scores(
        rf_test_scores, iso_test_scores, rf_weight, iso_weight)

    rf_pred     = (rf_test_scores     > rf_threshold).astype(int)
    iso_pred    = (iso_test_scores    > iso_threshold).astype(int)
    hybrid_pred = (hybrid_test_scores > hybrid_threshold).astype(int)

    evaluate_model(y_test, rf_pred,     f"{name} — SUPERVISED MODEL (RF)")
    evaluate_model(y_test, iso_pred,    f"{name} — UNSUPERVISED MODEL (ISO)")
    evaluate_model(y_test, hybrid_pred, f"{name} — OPTIMIZED HYBRID IDS")

  
    model_dir = f"{MODEL_DIR_BASE}/{name.lower()}"
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump({
        "rf_model":         rf,
        "iso_model":        iso,
        "scaler":           scaler,
        "rf_weight":        rf_weight,
        "iso_weight":       iso_weight,
        "rf_threshold":     rf_threshold,
        "iso_threshold":    iso_threshold,
        "hybrid_threshold": hybrid_threshold,
        "feature_columns":  feature_columns,
        "iso_min":          iso_min,
        "iso_max":          iso_max,
    }, f"{model_dir}/results.pkl")

    return joblib.load(f"{model_dir}/results.pkl")



def global_weighted_scoring(cicids_h, cert_h, hdfs_h):
    final_score = (GLOBAL_WEIGHTS["CICIDS"] * cicids_h +
                   GLOBAL_WEIGHTS["CERT"]   * cert_h   +
                   GLOBAL_WEIGHTS["HDFS"]   * hdfs_h)
    final_pred  = "ATTACK" if final_score > GLOBAL_THRESHOLD else "NORMAL"

    print("\n" + "="*70)
    print("  GLOBAL ENSEMBLE SCORING (Weighted Sum of 3 Hybrids)")
    print("="*70)
    print(f"  CICIDS Hybrid Score : {cicids_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['CICIDS']})")
    print(f"  CERT   Hybrid Score : {cert_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['CERT']})")
    print(f"  HDFS   Hybrid Score : {hdfs_h*100:6.2f}%  (Weight: {GLOBAL_WEIGHTS['HDFS']})")
    print("-"*50)
    print(f"  FINAL GLOBAL SCORE  : {final_score*100:.2f}%")
    print(f"  Global Threshold    : {GLOBAL_THRESHOLD}")
    print(f"  FINAL PREDICTION    : {final_pred}")
    print("="*70)
    return final_score, final_pred



def load_model(name):
    model_path = f"{MODEL_DIR_BASE}/{name.lower()}/results.pkl"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    return joblib.load(model_path)



def compute_global_metrics_test_split():
    """
    Evaluates each dataset's hybrid model on its test split using the
    per-dataset tuned hybrid_threshold (NOT the global 0.5 threshold).
    Aggregates into weighted global metrics.
    """
    print("\n" + "="*90)
    print("  GLOBAL ENSEMBLE PERFORMANCE METRICS  (TEST SPLIT)")
    print("="*90)


    cicids = load_model("CICIDS")
    cert   = load_model("CERT")
    hdfs   = load_model("HDFS")

 
    df_cicids = pd.read_csv(CICIDS_PATH)
    df_cert   = pd.read_csv(CERT_PATH)
    df_hdfs   = pd.read_csv(HDFS_PATH)


    X_cicids, y_cicids = preprocess_data(df_cicids.copy(), "label", "CICIDS")
    X_cert,   y_cert   = preprocess_data(df_cert.copy(),   "label", "CERT")
    X_hdfs,   y_hdfs   = preprocess_data(df_hdfs.copy(),   "label", "HDFS")

    
    def get_test_split(X, y):
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
        _, X_test, _, y_test = train_test_split(
            X_temp, y_temp, test_size=0.80, stratify=y_temp, random_state=RANDOM_STATE)
        return X_test, y_test

    X_test_c,  y_test_c  = get_test_split(X_cicids, y_cicids)
    X_test_ce, y_test_ce = get_test_split(X_cert,   y_cert)
    X_test_h,  y_test_h  = get_test_split(X_hdfs,   y_hdfs)

  
    def get_hybrid_on_test(model, X_test):
        rf_s  = get_rf_scores(model["rf_model"], X_test)
        iso_s = get_iso_scores(
            model["iso_model"], model["scaler"], X_test,
            model["iso_min"],   model["iso_max"])
        return get_hybrid_scores(rf_s, iso_s, model["rf_weight"], model["iso_weight"])

    cicids_hybrid = get_hybrid_on_test(cicids, X_test_c)
    cert_hybrid   = get_hybrid_on_test(cert,   X_test_ce)
    hdfs_hybrid   = get_hybrid_on_test(hdfs,   X_test_h)

 
    def evaluate_hybrid(model, hybrid_scores, y_true, name):
        tuned_th = model["hybrid_threshold"]
        pred     = (hybrid_scores > tuned_th).astype(int)

        acc  = accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, zero_division=0)
        rec  = recall_score(y_true, pred, zero_division=0)
        f1   = f1_score(y_true, pred, zero_division=0)
        cm   = confusion_matrix(y_true, pred)
        tn, fp, fn, tp = cm.ravel()

        print(f"\n  {name} Hybrid  |  Threshold: {tuned_th:.4f} (tuned on validation set)")
        print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
        print(f"  Accuracy  : {round(acc,  4)}")
        print(f"  Precision : {round(prec, 4)}")
        print(f"  Recall    : {round(rec,  4)}")
        print(f"  F1 Score  : {round(f1,   4)}")

        return acc, prec, rec, f1

    print("-"*90)
    acc_c,  prec_c,  rec_c,  f1_c  = evaluate_hybrid(cicids, cicids_hybrid, y_test_c,  "CICIDS")
    acc_ce, prec_ce, rec_ce, f1_ce = evaluate_hybrid(cert,   cert_hybrid,   y_test_ce, "CERT")
    acc_h,  prec_h,  rec_h,  f1_h  = evaluate_hybrid(hdfs,   hdfs_hybrid,   y_test_h,  "HDFS")

 
    W = GLOBAL_WEIGHTS
    weighted_acc  = W["CICIDS"]*acc_c  + W["CERT"]*acc_ce  + W["HDFS"]*acc_h
    weighted_prec = W["CICIDS"]*prec_c + W["CERT"]*prec_ce + W["HDFS"]*prec_h
    weighted_rec  = W["CICIDS"]*rec_c  + W["CERT"]*rec_ce  + W["HDFS"]*rec_h
    weighted_f1   = W["CICIDS"]*f1_c   + W["CERT"]*f1_ce   + W["HDFS"]*f1_h

    print("\n" + "="*60)
    print("  WEIGHTED GLOBAL ENSEMBLE SUMMARY")
    print("="*60)
    print(f"  Weighted Average Accuracy  : {round(weighted_acc,  4)}")
    print(f"  Weighted Average Precision : {round(weighted_prec, 4)}")
    print(f"  Weighted Average Recall    : {round(weighted_rec,  4)}")
    print(f"  Weighted Average F1 Score  : {round(weighted_f1,   4)}")
    print(f"\n  Thresholds Used:")
    print(f"    CICIDS : {cicids['hybrid_threshold']:.4f}")
    print(f"    CERT   : {cert['hybrid_threshold']:.4f}")
    print(f"    HDFS   : {hdfs['hybrid_threshold']:.4f}")
    print("="*90)

    return {
        "weighted_accuracy":  round(weighted_acc,  4),
        "weighted_precision": round(weighted_prec, 4),
        "weighted_recall":    round(weighted_rec,  4),
        "weighted_f1":        round(weighted_f1,   4),
    }



cicids_results = run_pipeline(CICIDS_PATH, "CICIDS")
cert_results   = run_pipeline(CERT_PATH,   "CERT")
hdfs_results   = run_pipeline(HDFS_PATH,   "HDFS")

print("\nAll models trained successfully!")


cicids_h = 0.2233
cert_h   = 0.6267
hdfs_h   = 0.5039
global_weighted_scoring(cicids_h, cert_h, hdfs_h)


global_metrics = compute_global_metrics_test_split()
print("\n✅ Global Ensemble Evaluation Completed!")


  RUNNING CICIDS PIPELINE

===== CICIDS =====

Label Distribution:
label
1    5000
0    5000
Name: count, dtype: int64

  Train=7500 | Val=500 | Test=2000

Training Random Forest...

Training Isolation Forest ONLY on NORMAL traffic
Normal Samples Used: 3750

  Best RF Threshold     : 0.2455
  Best ISO Threshold    : 0.3182
  Best Hybrid Threshold : 0.2859
  Best RF Weight        : 0.71
  Best ISO Weight       : 0.29

CICIDS — SUPERVISED MODEL (RF)
  TN=997  FP=3  FN=2  TP=998
  Accuracy  : 0.9975
  Precision : 0.997
  Recall    : 0.998
  F1 Score  : 0.9975

  Confusion Matrix:
  [[997   3]
 [  2 998]]

CICIDS — UNSUPERVISED MODEL (ISO)
  TN=894  FP=106  FN=149  TP=851
  Accuracy  : 0.8725
  Precision : 0.8892
  Recall    : 0.851
  F1 Score  : 0.8697

  Confusion Matrix:
  [[894 106]
 [149 851]]

CICIDS — OPTIMIZED HYBRID IDS
  TN=996  FP=4  FN=2  TP=998
  Accuracy  : 0.997
  Precision : 0.996
  Recall    : 0.998
  F1 Score  : 0.997

  Confusion Matrix:
  [[996   4]
 [  2 998]]

  RUNN